# 教程: 模型的预热期 (Warm-up Period)

## 目的
在运行动态模型时，我们通常无法精确地知道系统在模拟开始那一刻的“真实”初始状态（Initial State），例如，我们不知道流域的初始土壤含水量是多少。如果我们从一个不切实际的初始状态（如，假设土壤完全干燥）开始模拟，那么模拟初期的结果可能会有很大的偏差，这种偏差会持续一段时间，直到模型的状态在输入数据的驱动下逐渐调整到一个合理范围。

为了减小这种“初始条件效应”，一种常见的做法是设置一个**预热期（Warm-up or Spin-up Period）**。即，在我们的主要研究时段**之前**，先用一段历史或平均数据来运行模型，让模型的内部状态（如土壤含水量、地下水储量等）达到一个相对稳定和合理的值。然后，再从这个“预热”过的状态开始我们真正的模拟。

此笔记本将：
1.  通过两个情景对比，来展示预热期的作用。
2.  情景一：无预热期，从一个任意的初始状态开始模拟。
3.  情景二：有预热期，先用一段“背景”数据运行模型，再进行主时段的模拟。
4.  比较两个情景的状态和输出，以说明预热期的重要性。

## 1. 环境设置和模型准备

我们使用`SimpleRunoffModule`和`SimpleRouting`组成的模型，因为它有明确的内部状态`S`（土壤水）和`Q_s`（慢速流蓄水），便于观察。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from hydro_model.model import HydrologicalModel
from hydro_model.runoff import SimpleRunoffModule
from hydro_model.routing import SimpleRouting

# --- 定义一个辅助函数来运行模型并记录状态 ---
def run_and_record_states(model, rainfall, pet):
    history = {'outflow': [], 'soil_storage': []}
    for i in range(len(rainfall)):
        outflow = model.run(rainfall[i], pet[i])
        history['outflow'].append(outflow)
        history['soil_storage'].append(model.runoff_module.S)
    return history

# --- 定义共享的输入和参数 ---
# 我们有一个主要暴雨事件，前面是预热期数据
warmup_rainfall = np.array([2, 1, 3, 0, 2, 1, 1, 0, 4, 1] * 3) # 30天的预热期降雨
main_event_rainfall = np.array([10, 25, 60, 40, 20, 10, 5, 2, 0, 0])
full_rainfall = np.concatenate([warmup_rainfall, main_event_rainfall])
full_pet = np.full_like(full_rainfall, 2.5)
params = {'S_max': 200, 'c_loss': 0.05, 'k_q': 0.7, 'k_s': 0.2}

## 2. 运行情景一: 无预热期

在这个情景中，我们创建一个全新的模型实例（其初始状态`S`为0），并直接用**主暴雨事件**的数据来驱动它。

In [ ]:
print("运行情景一 (无预热期)...")
# 每次都创建一个新模型，确保初始状态为0
model_no_warmup = HydrologicalModel(SimpleRunoffModule(**params), SimpleRouting(**params))
history_no_warmup = run_and_record_states(model_no_warmup, main_event_rainfall, np.full_like(main_event_rainfall, 2.5))
print("运行完毕。")

## 3. 运行情景二: 有预热期

在这个情景中，我们先用30天的预热期数据来运行模型。这使得模型的土壤水状态`S`可以从0演变到一个更合理、与气候背景相符的水平。然后，我们**在不重置模型状态的情况下**，继续用主暴雨事件的数据来运行。

In [ ]:
print("\n运行情景二 (有预热期)...")
model_with_warmup = HydrologicalModel(SimpleRunoffModule(**params), SimpleRouting(**params))

# --- 预热阶段 ---
print("  - 正在运行预热期...")
history_warmup = run_and_record_states(model_with_warmup, warmup_rainfall, np.full_like(warmup_rainfall, 2.5))
print(f"  - 预热结束。最终土壤水状态 S = {model_with_warmup.runoff_module.S:.2f}")

# --- 主事件阶段 (接着预热后的状态) ---
print("  - 正在运行主事件...")
history_with_warmup = run_and_record_states(model_with_warmup, main_event_rainfall, np.full_like(main_event_rainfall, 2.5))
print("运行完毕。")

## 4. 结果对比

我们将两个情景在**主暴雨事件期间**的土壤水状态和出口流量进行对比。

In [ ]:
timesteps = np.arange(len(main_event_rainfall))
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
fig.suptitle('Effect of Model Warm-up Period', fontsize=16)

# --- 对比土壤水状态 S ---
ax1.plot(timesteps, history_no_warmup['soil_storage'], 'r-s', label='Without Warm-up (starts at S=0)')
ax1.plot(timesteps, history_with_warmup['soil_storage'], 'g-^', label='With Warm-up (starts at S > 0)')
ax1.set_title('Soil Storage (S) during Main Event')
ax1.set_ylabel('Storage (mm)')
ax1.legend()
ax1.grid(True, linestyle='--')

# --- 对比出口流量 ---
ax2.plot(timesteps, history_no_warmup['outflow'], 'r-s', label='Without Warm-up')
ax2.plot(timesteps, history_with_warmup['outflow'], 'g-^', label='With Warm-up')
ax2.set_title('Outflow Hydrograph during Main Event')
ax2.set_xlabel('Time Step (days)')
ax2.set_ylabel('Discharge (mm)')
ax2.legend()
ax2.grid(True, linestyle='--')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 5. 结论

从上图可以清晰地看到：
- **状态差异**: 在主事件开始时（`t=0`），“有预热”情景的土壤水`S`已经达到了一个非零的水平，而“无预热”情景的`S`为0。这个初始状态的差异直接影响了后续的模拟。
- **产流差异**: 由于“有预热”情景的初始土壤更湿润，它在同样的降雨下产生了**更高**的径流。这体现在其出口流量过程线的洪峰更高，总径流量也更大。

这个实验有力地证明了，在进行水文模拟时，特别是对于事件模拟，一个合理的预热期是至关重要的。它有助于消除任意设定的初始条件所带来的不确定性，从而使我们对关键事件的模拟结果更具信心。在实际应用中，预热期的长度可以从几个月到几年不等，取决于模型的“记忆”时长和达到平衡所需的时间。